# Módulo 7 · `Epochs`: de señal continua a ventanas etiquetadas

**Objetivo:** cortar la señal en ventanas, ponerle a cada una su etiqueta (ictal, preictal,
interictal, tipo de crisis…) y obtener el array `(n_ventanas, n_canales, n_muestras)`
que alimentará tu red. Es **el** módulo clave para la fase 2 del TFM.

Dos formas de crear `Epochs`:

| Situación | Función | Ejemplo |
|---|---|---|
| Ventanas alrededor de **eventos** | `mne.Epochs(raw, events, tmin, tmax)` | −10 s a +10 s del inicio de cada crisis |
| Ventanas **regulares** a lo largo de todo el registro | `mne.make_fixed_length_epochs(raw, duration, overlap)` | ventanas de 4 s cada 2 s → dataset de ML |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from tusz_utils import read_tusz_raw, preprocess

mne.set_log_level("WARNING")
mne.viz.set_browser_backend("matplotlib")

raw = preprocess(read_tusz_raw("aaaaabqv_s001_t000"))   # TCP, 0.5-40 Hz, 200 Hz
print(raw, raw.annotations)

## 7.1 Epochs alrededor de eventos: el inicio de cada crisis

In [ ]:
events, event_id = mne.events_from_annotations(raw)
print(event_id, events)

ep_onset = mne.Epochs(raw, events, event_id=event_id, tmin=-5, tmax=10,
                      baseline=None, preload=True, reject_by_annotation=False)
ep_onset

- `tmin`, `tmax`: ventana relativa al evento (s).
- `baseline`: corrección de línea base (`(None, 0)` = restar la media pre-evento). En EEG clínico
  continuo normalmente `None`.
- `reject_by_annotation`: si `True`, descarta epochs que toquen anotaciones `BAD_`.

In [ ]:
print(ep_onset.get_data().shape, "→ (crisis, canales, muestras)")
fig = ep_onset.plot_image(picks="FP2-F4", sigma=0, vmin=-200, vmax=200, show=False)
plt.show()

`plot_image` dibuja cada epoch como una fila: se ve el inicio sincronizado de la punta-onda en t = 0
en las 3 crisis. Su **promedio** (`Evoked`) muestra lo común a todas:

In [ ]:
evoked = ep_onset.average()
fig = evoked.plot(picks=["FP1-F3", "FP2-F4", "C3-CZ", "CZ-C4"], spatial_colors=False,
                  titles="Promedio de las 3 crisis alineadas en su inicio", show=False)
plt.show()

> En potenciales evocados el promedio es la estrella. En crisis promediar pierde sentido (la fase
> de la punta-onda no está sincronizada entre crisis), así que para crisis trabajarás con las
> epochs individuales o con su **espectro** (módulo 8).

## 7.2 Ventanas regulares: el dataset de ML

Cortamos todo el registro en ventanas de **4 s** con paso de **2 s** (50 % de solape):

In [ ]:
WIN, STEP = 4.0, 2.0
epochs = mne.make_fixed_length_epochs(raw, duration=WIN, overlap=WIN - STEP, preload=True,
                                      reject_by_annotation=False)
X = epochs.get_data()
print(epochs)
print("X:", X.shape, "=", X.shape[0], "ventanas ×", X.shape[1], "canales ×", X.shape[2], "muestras (4 s × 200 Hz)")

### Etiquetar cada ventana
El inicio de cada ventana está en `epochs.events[:, 0]` (en muestras). Comparamos cada ventana con
los intervalos de crisis:

In [ ]:
sf = raw.info["sfreq"]
starts = epochs.events[:, 0] / sf - raw.first_time
seizures = [(a["onset"] - raw.first_time, a["onset"] - raw.first_time + a["duration"], a["description"])
            for a in raw.annotations]

def solape(s, e, a, b):
    return max(0.0, min(e, b) - max(s, a))

ictal_frac = np.array([max([solape(s, s + WIN, a, b) for a, b, _ in seizures] + [0]) / WIN for s in starts])
print("Ventanas con algo de crisis:", (ictal_frac > 0).sum(), "| totalmente en crisis:", (ictal_frac == 1).sum())

### `metadata`: la forma elegante de guardar etiquetas
`epochs.metadata` es un **DataFrame con una fila por epoch**. Viaja con las epochs (al guardar, al
seleccionar, al descartar) y permite **consultas tipo pandas**.

In [ ]:
epochs.metadata = pd.DataFrame({"start": starts, "ictal_frac": ictal_frac,
                                "label": np.where(ictal_frac > 0, "ictal", "interictal")})
epochs.metadata.head()

In [ ]:
ictal = epochs['label == "ictal"']
fondo = epochs["ictal_frac == 0"]
print(len(ictal), "ictales |", len(fondo), "sin crisis")

## 7.3 Preictal / interictal: el planteamiento de **predicción**

Para predecir hay que definir:
- **SOP** (*seizure occurrence period*) / **preictal**: ventana de tiempo antes de la crisis que
  etiquetamos como "se acerca una crisis" (típicamente 5-60 min).
- **SPH** (*seizure prediction horizon*): margen entre el fin del preictal y el inicio de la crisis
  (tiempo para avisar al paciente). Típico: 0-5 min.
- **Interictal**: lejos de cualquier crisis (p. ej. > 1-4 h). Las ventanas entre medias se descartan.

```
 ── interictal ──|  (descartado)  |──── preictal (SOP) ────|─ SPH ─|█ crisis █|─ postictal ─
```

`tusz_utils.label_windows` / `make_windows` implementan esto. Como este fichero solo dura 24 min y la
primera crisis empieza a los 3 min, usamos valores de juguete (preictal = 60 s, SPH = 5 s):

In [ ]:
from tusz_utils import make_windows
ep = make_windows(raw, win=4.0, step=2.0, preictal=60, sph=5, postictal=30)
ep.metadata.label.value_counts()

In [ ]:
md = ep.metadata
colors = {"interictal": "tab:gray", "preictal": "tab:orange", "ictal": "tab:red", "postictal": "tab:blue"}
plt.figure(figsize=(12, 1.8))
for lab, c in colors.items():
    s = md[md.label == lab].start
    plt.scatter(s, np.zeros(len(s)), c=c, s=12, marker="|", label=lab)
plt.yticks([]); plt.xlim(0, 450); plt.xlabel("s"); plt.legend(ncol=4, loc="upper right")
plt.title("Etiqueta de cada ventana (primeros 450 s)"); plt.show()

> ⚠️ **Realidad de TUSZ para predicción**: los ficheros duran como mucho 30 min. Según tu inventario
> (`summary.txt`), en *train* solo **40 crisis** tienen ≥ 30 min de registro previo sin crisis dentro de la
> misma sesión. Por eso conviene encadenar los `t000, t001…` de una sesión (están ordenados en el tiempo)
> y ser realista con el SOP. Es una decisión metodológica importante para la memoria.

## 7.4 Rechazo automático de epochs

`reject` descarta epochs cuyo **pico a pico** supere un umbral en algún canal; `flat`, las que tengan
algún canal casi plano. Vemos qué se descarta y por qué con `drop_log`:

In [ ]:
ep_r = make_windows(raw, win=4.0, step=2.0)
ep_r.drop_bad(reject=dict(eeg=600e-6), flat=dict(eeg=1e-6))
print(f"Quedan {len(ep_r)} de {len(ep)}")
fig = ep_r.plot_drop_log(show=False); plt.show()

In [ ]:
# ¿He tirado ventanas de crisis? (¡comprobar siempre!)
print("Ictales antes:", (ep.metadata.label == "ictal").sum(), "| después:", (ep_r.metadata.label == "ictal").sum())

## 7.5 Balanceo de clases
Con ~2 % de ventanas ictales, un modelo que diga siempre "no crisis" acierta el 98 %. Estrategias:

1. **Submuestrear** la clase mayoritaria (lo más simple; abajo).
2. **Sobremuestrear** la minoritaria con más solape: ventanas ictales con paso de 0.5 s, fondo con paso de 4 s.
3. Pesos en la función de pérdida (`class_weight` en sklearn, `pos_weight` en PyTorch).

MNE ayuda con `equalize_event_counts`, pero con metadata es más flexible hacerlo con índices:

In [ ]:
rng = np.random.default_rng(0)
i_ict = np.where(ep.metadata.label == "ictal")[0]
i_int = np.where(ep.metadata.label == "interictal")[0]
i_bal = np.sort(np.r_[i_ict, rng.choice(i_int, size=len(i_ict) * 3, replace=False)])
ep_bal = ep[i_bal]
ep_bal.metadata.label.value_counts()

## 7.6 Guardar y cargar
Las epochs con metadata se guardan en `-epo.fif` y se cargan intactas:

In [ ]:
from tusz_utils import SALIDAS
ep.save(SALIDAS / "aaaaabqv_s001_t000-epo.fif", overwrite=True)
ep2 = mne.read_epochs(SALIDAS / "aaaaabqv_s001_t000-epo.fif")
print(ep2, ep2.metadata.columns.tolist())

Y de ahí, a NumPy para PyTorch/sklearn:

In [ ]:
X = ep2.get_data(units="uV").astype(np.float32)        # (n, 22, 800)
y = (ep2.metadata.label == "ictal").astype(int).values
print(X.shape, X.dtype, y.mean().round(3))

---
## 🧪 Ejercicios

**E7.1** Crea epochs de 2 s **solo dentro de las crisis** usando `events_from_annotations(raw, chunk_duration=2)`
y `mne.Epochs(..., tmin=0, tmax=2 - 1/sf)`. ¿Cuántas salen?

**E7.2** Implementa el *sobremuestreo por solape* (estrategia 2 de 7.5): ventanas ictales con paso 0.5 s y
fondo con paso 4 s, en un único array `X, y`.

**E7.3** Encadena `aaaaadqe_s001_t001` (4 crisis) y compara cuántas ventanas preictales (SOP 60 s, SPH 5 s)
obtienes respecto a `aaaaabqv`.

### Solución E7.1

In [ ]:
ev_c, eid_c = mne.events_from_annotations(raw, chunk_duration=2.0)
ep_c = mne.Epochs(raw, ev_c, eid_c, tmin=0, tmax=2 - 1 / sf, baseline=None, preload=True)
print(len(ep_c), ep_c.get_data().shape)

### Solución E7.2

In [ ]:
def windows_step(raw, win, step):
    return make_windows(raw, win=win, step=step)

e_ict = windows_step(raw, 4.0, 0.5); e_ict = e_ict['label == "ictal"']
e_bg = windows_step(raw, 4.0, 4.0);  e_bg = e_bg['label == "interictal"']
X2 = np.concatenate([e_ict.get_data(), e_bg.get_data()])
y2 = np.r_[np.ones(len(e_ict)), np.zeros(len(e_bg))]
print(X2.shape, "proporción ictal:", y2.mean().round(3))

### Solución E7.3

In [ ]:
r3 = preprocess(read_tusz_raw("aaaaadqe_s001_t001"))
e3 = make_windows(r3, preictal=60, sph=5)
pd.DataFrame({"aaaaabqv": ep.metadata.label.value_counts(), "aaaaadqe": e3.metadata.label.value_counts()})